## Risk model

In [3]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

In [4]:
df = pd.read_csv('Data/CFPB_Labeled_Complaints_FilledOnly.csv').drop_duplicates()

In [5]:
embedder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
print(df.shape)
print(df["Narrative"].isna().sum())

(688, 3)
0


In [7]:

embeddings = embedder.encode(
    df["Narrative"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

: 

In [168]:
embeddings.shape

(463, 768)

In [191]:

# X = embeddings
X = embeddings

# Three targets together
y = df[["Severity", "Vulnerability"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

regressor = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    verbose=0
)

model = MultiOutputRegressor(regressor)

model.fit(X_train, y_train)

preds = model.predict(X_test)

# MAE for each target
severity_mae = mean_absolute_error(
    y_test["Severity"],
    preds[:, 0]
)

vulnerability_mae = mean_absolute_error(
    y_test["Vulnerability"],
    preds[:, 1]
)


print("Severity MAE:", severity_mae)
print("Vulnerability MAE:", vulnerability_mae)

# Overall MAE
overall_mae = np.mean([
    severity_mae,
    vulnerability_mae
])

print("Overall MAE:", overall_mae)

ValueError: Found input variables with inconsistent numbers of samples: [463, 688]

In [188]:
df[['Severity']].value_counts()

Severity
1           135
2           124
5           119
4           116
3            99
0            95
Name: count, dtype: int64

In [189]:
df['Vulnerability'].value_counts()

Vulnerability
1    180
3    108
0    105
2    100
5     98
4     97
Name: count, dtype: int64

In [190]:
df[
    ["Severity",
     "Vulnerability"]
].corr(method="spearman")

,Severity,Vulnerability
Severity,1.000000,0.315626
Vulnerability,0.315626,1.000000


In [161]:
from scipy.stats import spearmanr

spearmanr(
    y_test["Vulnerability"],
    preds[:,0]
).statistic

np.float64(0.2611034454593457)

In [162]:
true_risk = (
    0.7 * y_test["Severity"]
    + 0.3 * y_test["Vulnerability"]
)

pred_risk = (
    0.7 * preds[:,0]
    + 0.3 * preds[:,1]
)

In [163]:
risk_mae = mean_absolute_error(
    true_risk,
    pred_risk
)

risk_r2 = r2_score(
    true_risk,
    pred_risk
)

risk_rmse = root_mean_squared_error(
    true_risk,
    pred_risk
)

print("Risk MAE:", risk_mae)
print("Risk RMSE:", risk_rmse)
print("Risk R²:", risk_r2)

Risk MAE: 0.7255687336049973
Risk RMSE: 0.900558791923135
Risk R²: 0.4391926007547756


In [164]:
text = 'My credit report is a mixed file.'
values = model.predict(embedder.encode([text]))
values

array([[3.11204468, 1.96098257]])

In [150]:
import joblib

joblib.dump(model, "Models/risk_model.joblib")

['Models/risk_model.joblib']

In [2]:
pip install tensorflow

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 MB 27.4 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 23.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 10.7 MB/s  0:00:00
Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
Using cached google_pasta-0.2.0-py3-none-any.whl (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.8 MB/s  0:00:00
Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl (25.8 MB)
Using cached opt_einsum-3.4.0-py3-none-any.whl (71 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [te